# 123. Best Time to Buy and Sell Stock III

## Topic Alignment
- **Role Relevance**: Multi-stage optimization with constraints appears in resource allocation across time windows, budget planning for ML experiments, and batch processing optimization.
- **Scenario**: Similar to scheduling two non-overlapping training jobs to maximize total performance gain, or allocating compute budget across two separate projects with time-dependent costs.

## Metadata Summary
- Source: [LeetCode - Best Time to Buy and Sell Stock III](https://leetcode.com/problems/best-time-to-buy-and-sell-stock-iii/)
- Tags: `Array`, `Dynamic Programming`, `Greedy`
- Difficulty: Hard
- Recommended Priority: High

## Problem Statement
You are given an array `prices` where `prices[i]` is the price of a given stock on the `i-th` day.

Find the maximum profit you can achieve. You may complete **at most two transactions**.

**Note**: You may not engage in multiple transactions simultaneously (i.e., you must sell the stock before you buy again).

**Constraints:**
- `1 <= prices.length <= 10^5`
- `0 <= prices[i] <= 10^5`

## Progressive Hints
- Hint 1: Think about states: you can be in one of several states (no stock, holding first stock, sold first, holding second, sold second).
- Hint 2: Use dynamic programming with state transitions. Track the maximum profit at each state for each day.
- Hint 3: You can optimize space by using only variables instead of arrays, since you only need the previous day's state.
- Hint 4: The key insight: for each day, decide whether to buy/sell/hold for each of the two possible transactions.
- Hint 5: Alternative approach: divide the array at each point and calculate max profit for one transaction before and after the split.

## Solution Overview
We can solve this with dynamic programming using state machines:

**State Machine DP Approach**:
Define 4 states:
1. `buy1`: Maximum profit after buying first stock
2. `sell1`: Maximum profit after selling first stock
3. `buy2`: Maximum profit after buying second stock
4. `sell2`: Maximum profit after selling second stock

**State transitions** for each day:
- `buy1 = max(buy1, -price)` (either already bought or buy today)
- `sell1 = max(sell1, buy1 + price)` (either already sold or sell today)
- `buy2 = max(buy2, sell1 - price)` (either already bought second or buy today)
- `sell2 = max(sell2, buy2 + price)` (either already sold second or sell today)

**Alternative Approach** (Two-pass):
1. Calculate max profit for one transaction ending at or before each day (left to right).
2. Calculate max profit for one transaction starting at or after each day (right to left).
3. Combine: `max(left[i] + right[i+1])` for all split points.

Both approaches are O(n) time and can be optimized to O(1) space.

## Detailed Explanation
### State Machine Approach (Optimal):

We maintain 4 state variables representing the maximum profit in each state:

**States and their meanings**:
1. `buy1`: Maximum profit after buying the first stock (negative value initially)
2. `sell1`: Maximum profit after completing the first transaction (buy then sell)
3. `buy2`: Maximum profit after buying the second stock (using profit from first transaction)
4. `sell2`: Maximum profit after completing both transactions

**Initialization**:
- `buy1 = -infinity` (haven't bought yet)
- `sell1 = 0` (no transaction yet)
- `buy2 = -infinity` (haven't bought second yet)
- `sell2 = 0` (no second transaction yet)

**For each price, update states in order**:
```python
buy1 = max(buy1, -price)
sell1 = max(sell1, buy1 + price)
buy2 = max(buy2, sell1 - price)
sell2 = max(sell2, buy2 + price)
```

**Why this order matters**: Each state depends on the previous state, so we update in sequence to ensure we use the correct previous values.

**Example walkthrough** for `prices = [3,3,5,0,0,3,1,4]`:

| Day | Price | buy1 | sell1 | buy2 | sell2 |
|-----|-------|------|-------|------|-------|
| -   | -     | -∞   | 0     | -∞   | 0     |
| 0   | 3     | -3   | 0     | -3   | 0     |
| 1   | 3     | -3   | 0     | -3   | 0     |
| 2   | 5     | -3   | 2     | -3   | 2     |
| 3   | 0     | 0    | 2     | 2    | 2     |
| 4   | 0     | 0    | 2     | 2    | 2     |
| 5   | 3     | 0    | 2     | 2    | 5     |
| 6   | 1     | 0    | 2     | 2    | 5     |
| 7   | 4     | 0    | 2     | 2    | 6     |

Result: 6 (buy at 3, sell at 5 for profit 2, then buy at 0, sell at 4 for profit 4)

### Two-Pass Approach (Alternative):

**Forward pass**: For each index i, calculate the maximum profit from one transaction in `prices[0:i+1]`
```python
left_profit[i] = max profit buying and selling in prices[0:i+1]
```

**Backward pass**: For each index i, calculate the maximum profit from one transaction in `prices[i:]`
```python
right_profit[i] = max profit buying and selling in prices[i:]
```

**Combine**: Maximum profit is `max(left_profit[i] + right_profit[i+1])` for all valid splits.

This approach is more intuitive but requires O(n) space unless carefully optimized.

## Complexity Trade-off Table
| Approach | Time Complexity | Space Complexity | Notes |
| --- | --- | --- | --- |
| State machine DP | O(n) | O(1) | Optimal solution. Clean and elegant. |
| Two-pass DP | O(n) | O(n) | More intuitive but uses extra space. |
| Two-pass optimized | O(n) | O(1) | Combine passes cleverly to avoid arrays. |
| Brute force | O(n^4) | O(1) | Try all pairs of transactions. Too slow. |
| General k transactions | O(kn) | O(k) | Generalizes to LC 188 with k transactions. |

## Reference Implementation

In [ ]:
from typing import List


def max_profit(prices: List[int]) -> int:
    """
    Find maximum profit with at most 2 transactions.
    
    Args:
        prices: Daily stock prices
    
    Returns:
        Maximum profit achievable with at most 2 transactions
    """
    if not prices:
        return 0
    
    # State machine approach
    # buy1: max profit after first buy
    # sell1: max profit after first sell
    # buy2: max profit after second buy
    # sell2: max profit after second sell
    
    buy1 = buy2 = float('-inf')
    sell1 = sell2 = 0
    
    for price in prices:
        # First transaction
        buy1 = max(buy1, -price)  # Buy at current price or keep previous buy
        sell1 = max(sell1, buy1 + price)  # Sell at current price or keep previous sell
        
        # Second transaction
        buy2 = max(buy2, sell1 - price)  # Buy second using profit from first
        sell2 = max(sell2, buy2 + price)  # Sell second at current price
    
    return sell2


def max_profit_two_pass(prices: List[int]) -> int:
    """
    Alternative two-pass solution for comparison.
    """
    if not prices or len(prices) < 2:
        return 0
    
    n = len(prices)
    
    # Forward pass: max profit with one transaction ending at or before i
    left_profit = [0] * n
    min_price = prices[0]
    for i in range(1, n):
        min_price = min(min_price, prices[i])
        left_profit[i] = max(left_profit[i-1], prices[i] - min_price)
    
    # Backward pass: max profit with one transaction starting at or after i
    right_profit = [0] * n
    max_price = prices[-1]
    for i in range(n-2, -1, -1):
        max_price = max(max_price, prices[i])
        right_profit[i] = max(right_profit[i+1], max_price - prices[i])
    
    # Combine: find best split point
    max_profit_total = 0
    for i in range(n):
        max_profit_total = max(max_profit_total, left_profit[i] + right_profit[i])
    
    return max_profit_total

## Validation

In [ ]:
cases = [
    ([3, 3, 5, 0, 0, 3, 1, 4], 6),  # Buy at 0, sell at 3, buy at 1, sell at 4
    ([1, 2, 3, 4, 5], 4),  # Single transaction: buy at 1, sell at 5
    ([7, 6, 4, 3, 1], 0),  # No profit possible
    ([1], 0),  # Single day
    ([1, 2], 1),  # Single transaction
    ([2, 1, 2, 0, 1], 2),  # Two transactions of 1 each
    ([6, 1, 3, 2, 4, 7], 7),  # Buy at 1, sell at 7
    ([1, 2, 4, 2, 5, 7, 2, 4, 9, 0], 13),  # Two transactions: 1->7 (6) and 2->9 (7)
]

for prices, expected in cases:
    result1 = max_profit(prices.copy())
    result2 = max_profit_two_pass(prices.copy())
    assert result1 == expected, f"State machine failed for {prices}: got {result1}, expected {expected}"
    assert result2 == expected, f"Two-pass failed for {prices}: got {result2}, expected {expected}"

print('All tests passed for LC 123.')

## Complexity Analysis
### State Machine Approach:
- **Time Complexity**: O(n) where n is the number of days
  - Single pass through the prices array
  - Constant time operations per day (4 state updates)
- **Space Complexity**: O(1)
  - Only 4 state variables maintained
  - No additional data structures
- **Bottleneck**: The linear scan is unavoidable.

### Two-Pass Approach:
- **Time Complexity**: O(n)
  - Forward pass: O(n)
  - Backward pass: O(n)
  - Combine: O(n)
- **Space Complexity**: O(n) for the two arrays
  - Can be optimized to O(1) with careful implementation

## Edge Cases & Pitfalls
- **Single day**: No transaction possible, return 0.
- **Decreasing prices**: No profitable transaction, return 0.
- **One transaction optimal**: Second transaction not needed, algorithm handles naturally.
- **Non-overlapping optimal**: Two separate peaks give best profit.
- **Overlapping consideration**: Must sell before buying again (enforced by state transitions).
- **State update order**: In state machine approach, updating in wrong order breaks dependencies.
- **Initialization**: `buy1` and `buy2` must start at -infinity, not 0.
- **Integer overflow**: Not an issue in Python, but be careful in C++/Java.

## Follow-up Variants
- What if you can perform at most k transactions? → LC 188 (generalize this approach)
- What if k is very large (k >= n/2)? → Reduces to unlimited transactions (LC 122)
- What if there's a transaction fee? → Combine with LC 714 approach
- What if there's a cooldown between transactions? → Combine with LC 309 approach
- Can you track which days to buy/sell, not just the profit?

## Takeaways
- State machine DP is powerful for problems with distinct phases/stages.
- When transactions have limits, use state variables to track each transaction.
- The state update order matters when states depend on each other.
- Two-pass algorithms can often be replaced by single-pass with state tracking.
- For k transactions, this generalizes to O(kn) time and O(k) space.
- Always consider whether you need arrays or just variables (space optimization).
- This problem demonstrates how DP and greedy approaches can be combined: greedy local decisions within a DP framework.

## Similar Problems
| Problem ID | Problem Title | Technique |
| --- | --- | --- |
| 121 | Best Time to Buy and Sell Stock | One Pass Greedy |
| 122 | Best Time to Buy and Sell Stock II | Greedy |
| 188 | Best Time to Buy and Sell Stock IV | DP (k transactions) |
| 309 | Best Time to Buy and Sell Stock with Cooldown | DP State Machine |
| 714 | Best Time to Buy and Sell Stock with Transaction Fee | DP State Machine |